# V1-23: 재무 Tool 실행과 계산 근거 확인

## 이번 테스트의 목적

21번에서는 LLM의 Tool **요청서**만, 22번에서는 실제 조회할 **연도**만 확인했습니다. 이번에는 확정된 기업·사업연도를 입력으로 LangChain 재무 Tool을 직접 실행하고, OpenDART 원천값 선택과 계산 결과를 점검합니다.

- 9개 원천 항목: 매출액, 영업이익, 당기순이익, 자산·부채·자본, 유동자산·유동부채, 영업활동현금흐름
- 8개 계산 지표: 매출 성장률, 영업이익률, 순이익률, ROA, ROE, 부채비율, 유동비율, 영업현금흐름률
- 실패 경계: 금융업 제외, CFS 전체 부재 시 OFS 대체, 없는 사업연도, 결측·0/음수 분모, 서로 다른 기준·통화

LLM은 호출하지 않습니다. 평가 질문의 기업·연도는 이미 정해진 주문서이며, 이 단계는 **Tool 실행 결과와 결정론적 계산의 신뢰성**을 분리해 확인합니다. 최종 workflow와 운영 모듈은 작성하지 않습니다.

정답 금액 36개는 기존 04번 노트북의 저장된 OpenDART 원천 결과를 실행 전에 고정한 개발용 회귀 기준입니다. 02번의 주요 수치와도 일치합니다. 독립 holdout이나 모든 업종에 대한 일반화 근거는 아닙니다. 계산값은 이 원천 금액과 아래에 명시한 공식에서 미리 계산해 고정했습니다. 공시가 정정되어 접수번호가 바뀌면 해당 필드는 오답으로 세지 않고 원천 버전 변경으로 분리합니다.

자료: OpenDART 단일회사 전체 재무제표·기업개황 [공식 개발가이드](https://opendart.fss.or.kr/guide/detail.do?apiGrpCd=DS003&apiId=2019020), [CFA 비율 목록](https://www.cfainstitute.org/sites/default/files/-/media/documents/support/programs/cfa/cfa_program_level_ii_financial_ratio_list.pdf), 프로젝트 [도메인 레퍼런스](../../docs/references/domain/README.md). FinSight의 부채비율은 **부채총계 ÷ 자본총계**이며 CFA의 이자부차입금 기반 debt-to-equity와 정의가 다릅니다.

In [ ]:
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Literal
import json
import os
import sys
import time

import pandas as pd
from IPython.display import display
from langchain.tools import tool

In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

sys.path.insert(0, str(project_root / 'src'))
from dart.client import _get_bytes, fetch_financial_rows, find_corp_info

REPORT_CODE = '11011'
COMPANY_URL = 'https://opendart.fss.or.kr/api/company.json'
run_config = {'run_at_utc': datetime.now(timezone.utc).isoformat(), 'report_code': REPORT_CODE}
print(f'Project root: {project_root}')
print(f'OpenDART API key loaded: {bool(api_key)} (값은 출력하지 않음)')
run_config

In [ ]:
# 실행 전에 고정한 개발용 정답. 출처: 04번 저장 출력, 접수번호는 02번·20번에 기록.
METRICS = (
    'revenue', 'operating_income', 'net_income', 'assets', 'liabilities',
    'equity', 'current_assets', 'current_liabilities', 'operating_cash_flow',
)
GOLD = {
    ('삼성전자', 2023): {
        'rcept_no': '20240312000736', 'fs_div': 'CFS',
        'values': {
            'revenue': 258935494000000, 'operating_income': 6566976000000,
            'net_income': 15487100000000, 'assets': 455905980000000,
            'liabilities': 92228115000000, 'equity': 363677865000000,
            'current_assets': 195936557000000, 'current_liabilities': 75719452000000,
            'operating_cash_flow': 44137427000000,
        },
    },
    ('삼성전자', 2024): {
        'rcept_no': '20250311001085', 'fs_div': 'CFS',
        'values': {
            'revenue': 300870903000000, 'operating_income': 32725961000000,
            'net_income': 34451351000000, 'assets': 514531948000000,
            'liabilities': 112339878000000, 'equity': 402192070000000,
            'current_assets': 227062266000000, 'current_liabilities': 93326299000000,
            'operating_cash_flow': 72982621000000,
        },
    },
    ('SK하이닉스', 2023): {
        'rcept_no': '20240319000684', 'fs_div': 'CFS',
        'values': {
            'revenue': 32765719000000, 'operating_income': -7730313000000,
            'net_income': -9137547000000, 'assets': 100330165000000,
            'liabilities': 46826413000000, 'equity': 53503752000000,
            'current_assets': 30468100000000, 'current_liabilities': 21007810000000,
            'operating_cash_flow': 4278191000000,
        },
    },
    ('SK하이닉스', 2024): {
        'rcept_no': '20250319000665', 'fs_div': 'CFS',
        'values': {
            'revenue': 66192960000000, 'operating_income': 23467319000000,
            'net_income': 19796902000000, 'assets': 119855209000000,
            'liabilities': 45939505000000, 'equity': 73915704000000,
            'current_assets': 42278887000000, 'current_liabilities': 24965444000000,
            'operating_cash_flow': 29795885000000,
        },
    },
}

# 04번 저장 출력에서 금액과 별도로 확인한 계정 ID·재무제표 구분·통화 정답.
GOLD_ACCOUNT_ID = {
    'revenue': 'ifrs-full_Revenue', 'operating_income': 'dart_OperatingIncomeLoss',
    'net_income': 'ifrs-full_ProfitLoss', 'assets': 'ifrs-full_Assets',
    'liabilities': 'ifrs-full_Liabilities', 'equity': 'ifrs-full_Equity',
    'current_assets': 'ifrs-full_CurrentAssets', 'current_liabilities': 'ifrs-full_CurrentLiabilities',
    'operating_cash_flow': 'ifrs-full_CashFlowsFromUsedInOperatingActivities',
}
GOLD_SECTION = {
    ('삼성전자', 'revenue'): 'IS', ('삼성전자', 'operating_income'): 'IS', ('삼성전자', 'net_income'): 'IS',
    ('SK하이닉스', 'revenue'): 'CIS', ('SK하이닉스', 'operating_income'): 'CIS', ('SK하이닉스', 'net_income'): 'CIS',
}

GOLD_RATIO_2024 = {
    '삼성전자': {
        'revenue_growth': '16.1953', 'operating_margin': '10.8771',
        'net_margin': '11.4505', 'roa': '7.1002', 'roe': '8.9967',
        'debt_ratio': '27.9319', 'current_ratio': '243.2993',
        'operating_cash_flow_margin': '24.2571',
    },
    'SK하이닉스': {
        'revenue_growth': '102.0189', 'operating_margin': '35.4529',
        'net_margin': '29.9079', 'roa': '17.9820', 'roe': '31.0736',
        'debt_ratio': '62.1512', 'current_ratio': '169.3496',
        'operating_cash_flow_margin': '45.0137',
    },
}

REQUESTS = [
    {'id': 'T01', 'companies': ['삼성전자', 'SK하이닉스'], 'years': [2023, 2024], 'metrics': list(METRICS)},
    {'id': 'T02', 'companies': ['리노공업'], 'years': [2024], 'metrics': list(METRICS)},
    {'id': 'T03', 'companies': ['KB금융'], 'years': [2024], 'metrics': ['revenue']},
    {'id': 'T04', 'companies': ['삼성전자'], 'years': [2026], 'metrics': ['revenue']},
]
display(pd.DataFrame([{'사례': x['id'], '기업': x['companies'], '사업연도': x['years'], '요청 항목 수': len(x['metrics'])} for x in REQUESTS]))

In [ ]:
# 원천 계정 매핑 후보: account_id 우선, 이름은 보조. 손익은 IS 우선, 없으면 CIS.
ACCOUNT_RULES = {
    'revenue': {'sections': ('IS', 'CIS'), 'ids': ('ifrs-full_Revenue', 'ifrs-full_RevenueFromContractsWithCustomers'), 'names': ('매출액', '수익(매출액)', '영업수익')},
    'operating_income': {'sections': ('IS', 'CIS'), 'ids': ('dart_OperatingIncomeLoss',), 'names': ('영업이익', '영업이익(손실)')},
    'net_income': {'sections': ('IS', 'CIS'), 'ids': ('ifrs-full_ProfitLoss',), 'names': ('당기순이익', '당기순이익(손실)')},
    'assets': {'sections': ('BS',), 'ids': ('ifrs-full_Assets',), 'names': ('자산총계',)},
    'liabilities': {'sections': ('BS',), 'ids': ('ifrs-full_Liabilities',), 'names': ('부채총계',)},
    'equity': {'sections': ('BS',), 'ids': ('ifrs-full_Equity',), 'names': ('자본총계',)},
    'current_assets': {'sections': ('BS',), 'ids': ('ifrs-full_CurrentAssets',), 'names': ('유동자산',)},
    'current_liabilities': {'sections': ('BS',), 'ids': ('ifrs-full_CurrentLiabilities',), 'names': ('유동부채',)},
    'operating_cash_flow': {'sections': ('CF',), 'ids': ('ifrs-full_CashFlowsFromUsedInOperatingActivities',), 'names': ('영업활동으로 인한 현금흐름', '영업활동으로인한현금흐름', '영업활동 현금흐름', '영업활동현금흐름')},
}
display(pd.DataFrame([{'항목': key, '재무제표': rule['sections'], '우선 account_id': rule['ids']} for key, rule in ACCOUNT_RULES.items()]))

In [ ]:
company_cache, overview_cache, api_log = {}, {}, []


def company_info(name):
    if name not in company_cache:
        company_cache[name] = find_corp_info(name, api_key)
    return company_cache[name]


def company_overview(corp_code):
    if corp_code not in overview_cache:
        response = json.loads(_get_bytes(COMPANY_URL, {'crtfc_key': api_key, 'corp_code': corp_code}).decode('utf-8'))
        if response.get('status') != '000':
            raise RuntimeError(f"기업개황 오류 [{response.get('status')}]: {response.get('message')}")
        overview_cache[corp_code] = response
    return overview_cache[corp_code]


def v1_eligibility(overview):
    code = (overview.get('induty_code') or '').strip()
    if not code.isdigit():
        return 'UNKNOWN'
    return 'EXCLUDED_FINANCIAL' if code[:2] in {'64', '65', '66'} else 'ELIGIBLE'


def fetch_statement(corp_code, year, basis='DEFAULT'):
    divisions = ('CFS', 'OFS') if basis == 'DEFAULT' else (basis,)
    attempts = []
    for fs_div in divisions:
        started = time.perf_counter()
        try:
            rows = fetch_financial_rows(corp_code, year, api_key, REPORT_CODE, fs_div)
            status = 'AVAILABLE' if rows else 'EMPTY_RESPONSE'
            error = None
        except RuntimeError as exc:
            rows = []
            status = 'NO_DATA' if '[013]' in str(exc) else 'API_ERROR'
            error = str(exc)
        except Exception as exc:
            rows = []
            status = 'API_ERROR'
            error = f'{type(exc).__name__}: {exc}'
        attempt = {'corp_code': corp_code, 'year': year, 'fs_div': fs_div, 'status': status,
                   'row_count': len(rows), 'latency_ms': (time.perf_counter() - started) * 1000,
                   'error': error}
        api_log.append(attempt)
        attempts.append(attempt)
        if status == 'AVAILABLE':
            return {'status': 'AVAILABLE', 'fs_div': fs_div, 'rows': rows, 'attempts': attempts}
        if not (basis == 'DEFAULT' and fs_div == 'CFS' and status == 'NO_DATA'):
            return {'status': status, 'fs_div': None, 'rows': [], 'attempts': attempts}
    return {'status': 'NO_DATA', 'fs_div': None, 'rows': [], 'attempts': attempts}

In [ ]:
def parse_amount(raw):
    if raw is None or str(raw).strip() in {'', '-'}:
        return None, 'MISSING_AMOUNT'
    try:
        return int(str(raw).replace(',', '').strip()), 'OK'
    except ValueError:
        return None, 'INVALID_AMOUNT'


def select_field(rows, metric):
    rule = ACCOUNT_RULES[metric]
    for section in rule['sections']:
        section_rows = [row for row in rows if row.get('sj_div') == section]
        for key, expected, matched_by in (
            ('account_id', rule['ids'], 'account_id'),
            ('account_nm', rule['names'], 'account_nm'),
        ):
            matches = [row for row in section_rows if row.get(key) in expected]
            if len(matches) > 1:
                return {'status': 'AMBIGUOUS', 'amount': None, 'sj_div': section,
                        'matched_by': matched_by, 'candidate_count': len(matches)}
            if len(matches) == 1:
                row = matches[0]
                amount, status = parse_amount(row.get('thstrm_amount'))
                return {
                    'status': status, 'amount': amount, 'sj_div': section,
                    'account_id': row.get('account_id'), 'account_nm': row.get('account_nm'),
                    'currency': row.get('currency'), 'rcept_no': row.get('rcept_no'),
                    'matched_by': matched_by, 'candidate_count': 1,
                }
    return {'status': 'MISSING_ACCOUNT', 'amount': None, 'candidate_count': 0}

In [ ]:
@tool
def get_financial_data(
    companies: list[str],
    metrics: list[str],
    years: list[int] | None = None,
    recent_years: int | None = None,
    statement_basis: Literal['DEFAULT', 'CFS', 'OFS'] = 'DEFAULT',
) -> dict:
    """확정된 기업명·사업연도로 OpenDART 연간 재무 원천값을 조회한다. 최근 N년 요청은 별도 기간 Resolver에서 years로 확정한 뒤 실행한다."""
    if recent_years is not None or not years:
        return {'status': 'PERIOD_UNRESOLVED', 'records': []}
    if not companies or not metrics or any(metric not in ACCOUNT_RULES for metric in metrics):
        return {'status': 'INVALID_ARGUMENTS', 'records': []}

    records = []
    for name in companies:
        try:
            corp = company_info(name)
            overview = company_overview(corp['corp_code'])
        except LookupError as exc:
            records.append({'company': name, 'year': None, 'status': 'COMPANY_UNRESOLVED', 'reason': str(exc)})
            continue
        except Exception as exc:
            records.append({'company': name, 'year': None, 'status': 'API_ERROR', 'reason': str(exc)})
            continue

        eligibility = v1_eligibility(overview)
        if eligibility != 'ELIGIBLE':
            records.append({'company': name, 'year': None, 'status': eligibility,
                            'corp_code': corp['corp_code'], 'induty_code': overview.get('induty_code')})
            continue

        for year in years:
            result = fetch_statement(corp['corp_code'], year, statement_basis)
            if result['status'] != 'AVAILABLE':
                records.append({'company': name, 'year': year, 'status': result['status'],
                                'corp_code': corp['corp_code'], 'attempts': result['attempts']})
                continue
            fields = {metric: select_field(result['rows'], metric) for metric in metrics}
            records.append({
                'company': name, 'year': year, 'status': 'AVAILABLE' if all(x['status'] == 'OK' for x in fields.values()) else 'PARTIAL',
                'corp_code': corp['corp_code'], 'fs_div': result['fs_div'], 'induty_code': overview.get('induty_code'),
                'acc_mt': overview.get('acc_mt'), 'fields': fields, 'attempts': result['attempts'],
            })
    return {'status': 'DONE', 'records': records}


print('LangChain Tool:', get_financial_data.name)
print('입력 스키마:', get_financial_data.args)

In [ ]:
# 실제 OpenDART 호출은 이 셀에서만 일어난다. LLM은 호출하지 않는다.
tool_results = {}
for case in REQUESTS:
    print(f"[{case['id']}] {', '.join(case['companies'])} / {case['years']} 호출 시작")
    started = time.perf_counter()
    output = get_financial_data.invoke({
        'companies': case['companies'], 'metrics': case['metrics'], 'years': case['years'],
        'recent_years': None, 'statement_basis': 'DEFAULT',
    })
    tool_results[case['id']] = output
    print(f"  완료: {[(record['company'], record['year'], record['status']) for record in output['records']]} / {(time.perf_counter() - started):.2f}초")

records = [record for output in tool_results.values() for record in output['records']]
record_by_key = {(record['company'], record['year']): record for record in records}
display(pd.DataFrame([{
    '기업': record['company'], '사업연도': record.get('year'), '상태': record['status'],
    'CFS/OFS': record.get('fs_div'), '매핑 완료 항목 수': sum(x['status'] == 'OK' for x in record.get('fields', {}).values()),
    '업종코드': record.get('induty_code'),
} for record in records]))
if api_log:
    display(pd.DataFrame(api_log)[['corp_code', 'year', 'fs_div', 'status', 'row_count', 'latency_ms', 'error']])
else:
    print('재무 API 호출 기록이 없습니다. 기업 식별·개황 단계의 오류를 확인하세요.')

In [ ]:
# Field Accuracy: 04번에 저장된 계정·구분·통화·금액·접수번호와 Tool 출력을 비교한다.
field_checks = []
for (name, year), gold in GOLD.items():
    record = record_by_key.get((name, year), {})
    for metric, expected_amount in gold['values'].items():
        field = record.get('fields', {}).get(metric, {})
        source_same = field.get('rcept_no') == gold['rcept_no']
        expected_section = GOLD_SECTION.get((name, metric), 'CF' if metric == 'operating_cash_flow' else 'BS')
        state = 'SOURCE_CHANGED' if field.get('rcept_no') and not source_same else (
            'MATCH' if record.get('fs_div') == gold['fs_div'] and field.get('status') == 'OK'
            and field.get('amount') == expected_amount and field.get('account_id') == GOLD_ACCOUNT_ID[metric]
            and field.get('sj_div') == expected_section and field.get('currency') == 'KRW'
            and source_same else 'MISMATCH'
        )
        field_checks.append({
            '기업': name, '사업연도': year, '항목': metric, '판정': state,
            '기대 금액': expected_amount, '실제 금액': field.get('amount'),
            '기대 접수번호': gold['rcept_no'], '실제 접수번호': field.get('rcept_no'),
            '기대 account_id': GOLD_ACCOUNT_ID[metric], '실제 account_id': field.get('account_id'),
            '기대 sj_div': expected_section, '실제 sj_div': field.get('sj_div'), '실제 통화': field.get('currency'),
        })

field_frame = pd.DataFrame(field_checks)
scored_fields = field_frame[field_frame['판정'] != 'SOURCE_CHANGED']
field_accuracy = (scored_fields['판정'] == 'MATCH').mean() if not scored_fields.empty else None
print(f"Field Accuracy: {int((scored_fields['판정'] == 'MATCH').sum())}/{len(scored_fields)} = {field_accuracy}")
print(f"원천 버전 변경으로 평가 제외: {(field_frame['판정'] == 'SOURCE_CHANGED').sum()}건")
display(field_frame[field_frame['판정'] != 'MATCH'])

expected_records = {
    ('삼성전자', 2023): ('AVAILABLE', 'CFS'),
    ('삼성전자', 2024): ('AVAILABLE', 'CFS'),
    ('SK하이닉스', 2023): ('AVAILABLE', 'CFS'),
    ('SK하이닉스', 2024): ('AVAILABLE', 'CFS'),
    ('리노공업', 2024): ('AVAILABLE', 'OFS'),
    ('KB금융', None): ('EXCLUDED_FINANCIAL', None),
    ('삼성전자', 2026): ('NO_DATA', None),
}
contract_checks = [{
    '기업·연도': key,
    '기대 상태/기준': expected,
    '실제 상태/기준': (record_by_key.get(key, {}).get('status'), record_by_key.get(key, {}).get('fs_div')),
    '일치': (record_by_key.get(key, {}).get('status'), record_by_key.get(key, {}).get('fs_div')) == expected,
} for key, expected in expected_records.items()]
display(pd.DataFrame(contract_checks))
print(f"상태·기준 계약 일치: {sum(x['일치'] for x in contract_checks)}/{len(contract_checks)}")

In [ ]:
# 2024년 8개 계산식. 금액은 정수 원화, 비율만 소수점 네 자리에서 반올림한다.
FORMULAS = {
    'revenue_growth': '(당기 매출액 - 전기 매출액) / 전기 매출액 × 100',
    'operating_margin': '영업이익 / 매출액 × 100',
    'net_margin': '당기순이익 / 매출액 × 100',
    'roa': '당기순이익 / 평균(전기말·당기말 자산총계) × 100',
    'roe': '당기순이익 / 평균(전기말·당기말 자본총계) × 100',
    'debt_ratio': '부채총계 / 자본총계 × 100',
    'current_ratio': '유동자산 / 유동부채 × 100',
    'operating_cash_flow_margin': '영업활동현금흐름 / 매출액 × 100',
}
PREVIOUS_REQUIRED = {'revenue_growth', 'roa', 'roe'}
REQUIRED_FIELDS = {
    'revenue_growth': (('revenue',), ('revenue',)),
    'operating_margin': (('operating_income', 'revenue'), ()),
    'net_margin': (('net_income', 'revenue'), ()),
    'roa': (('net_income', 'assets'), ('assets',)),
    'roe': (('net_income', 'equity'), ('equity',)),
    'debt_ratio': (('liabilities', 'equity'), ()),
    'current_ratio': (('current_assets', 'current_liabilities'), ()),
    'operating_cash_flow_margin': (('operating_cash_flow', 'revenue'), ()),
}


def calculate_metric(current, previous, metric):
    if current is None or current.get('status') != 'AVAILABLE':
        return {'status': 'MISSING_RECORD', 'value_pct': None}
    if metric in PREVIOUS_REQUIRED:
        if previous is None or previous.get('status') != 'AVAILABLE':
            return {'status': 'MISSING_PREVIOUS', 'value_pct': None}
        if current.get('fs_div') != previous.get('fs_div'):
            return {'status': 'BASIS_MISMATCH', 'value_pct': None}

    current_keys, previous_keys = REQUIRED_FIELDS[metric]
    selected_fields = [current.get('fields', {}).get(key, {}) for key in current_keys]
    selected_fields += [previous.get('fields', {}).get(key, {}) for key in previous_keys]
    if any(field.get('status') != 'OK' or field.get('amount') is None for field in selected_fields):
        return {'status': 'MISSING_INPUT', 'value_pct': None}
    currencies = {field.get('currency') for field in selected_fields}
    if len(currencies) != 1 or None in currencies:
        return {'status': 'CURRENCY_MISMATCH', 'value_pct': None}

    c = {key: Decimal(current['fields'][key]['amount']) for key in current_keys}
    p = {key: Decimal(previous['fields'][key]['amount']) for key in previous_keys}
    if metric == 'revenue_growth':
        numerator, denominator = c['revenue'] - p['revenue'], p['revenue']
    elif metric == 'operating_margin':
        numerator, denominator = c['operating_income'], c['revenue']
    elif metric == 'net_margin':
        numerator, denominator = c['net_income'], c['revenue']
    elif metric == 'roa':
        numerator, denominator = c['net_income'], (c['assets'] + p['assets']) / 2
    elif metric == 'roe':
        numerator, denominator = c['net_income'], (c['equity'] + p['equity']) / 2
    elif metric == 'debt_ratio':
        numerator, denominator = c['liabilities'], c['equity']
    elif metric == 'current_ratio':
        numerator, denominator = c['current_assets'], c['current_liabilities']
    else:
        numerator, denominator = c['operating_cash_flow'], c['revenue']
    if denominator <= 0:
        return {'status': 'INVALID_DENOMINATOR', 'value_pct': None,
                'formula': FORMULAS[metric], 'denominator': str(denominator)}

    value = (numerator / denominator * 100).quantize(Decimal('0.0001'), rounding=ROUND_HALF_UP)
    return {'status': 'OK', 'value_pct': str(value), 'formula': FORMULAS[metric],
            'numerator': str(numerator), 'denominator': str(denominator),
            'currency': next(iter(currencies)), 'fs_div': current['fs_div']}

In [ ]:
# 실제 OpenDART 조회 결과로 계산한 16개 비율을 고정 정답과 비교한다. 정정 공시 입력은 별도 판정한다.
field_state = {(row['기업'], row['사업연도'], row['항목']): row['판정'] for row in field_checks}
calculation_rows = []
for company, expected_metrics in GOLD_RATIO_2024.items():
    current = record_by_key.get((company, 2024))
    previous = record_by_key.get((company, 2023))
    for metric, expected in expected_metrics.items():
        current_keys, previous_keys = REQUIRED_FIELDS[metric]
        source_changed = any(field_state.get((company, 2024, key)) == 'SOURCE_CHANGED' for key in current_keys)
        source_changed |= any(field_state.get((company, 2023, key)) == 'SOURCE_CHANGED' for key in previous_keys)
        actual = calculate_metric(current, previous, metric)
        evaluation_state = 'SOURCE_CHANGED' if source_changed else actual['status']
        calculation_rows.append({
            '기업': company, '사업연도': 2024, '지표': metric, '공식': FORMULAS[metric],
            '기대값(%)': expected, '계산값(%)': actual['value_pct'],
            '상태': evaluation_state, '일치': evaluation_state == 'OK' and actual['value_pct'] == expected,
            '분자': actual.get('numerator'), '분모': actual.get('denominator'),
        })

calculation_frame = pd.DataFrame(calculation_rows)
display(calculation_frame[['기업', '사업연도', '지표', '기대값(%)', '계산값(%)', '상태', '일치']])
scored_calculations = calculation_frame[calculation_frame['상태'] != 'SOURCE_CHANGED']
calculation_accuracy = scored_calculations['일치'].mean() if not scored_calculations.empty else None
print(f"Calculation Accuracy: {int(scored_calculations['일치'].sum())}/{len(scored_calculations)} = {calculation_accuracy}")
print(f"원천 버전 변경으로 평가 제외: {(calculation_frame['상태'] == 'SOURCE_CHANGED').sum()}건")
if not scored_calculations['일치'].all():
    print('실패 계산의 공식·분자·분모를 확인하세요.')
    display(scored_calculations[~scored_calculations['일치']])

In [ ]:
# 실제 응답에 없을 수 있는 위험 상황은 작은 고정 fixture로 확인한다.
def fixture(values, basis='CFS', currency='KRW', status='AVAILABLE'):
    return {'status': status, 'fs_div': basis,
            'fields': {key: {'status': 'OK' if value is not None else 'MISSING_AMOUNT',
                             'amount': value, 'currency': currency} for key, value in values.items()}}


UNSAFE_CASES = [
    ('매출 0', fixture({'operating_income': 5, 'revenue': 0}), None, 'operating_margin', 'INVALID_DENOMINATOR'),
    ('자본 음수', fixture({'liabilities': 10, 'equity': -5}), None, 'debt_ratio', 'INVALID_DENOMINATOR'),
    ('유동부채 결측', fixture({'current_assets': 10, 'current_liabilities': None}), None, 'current_ratio', 'MISSING_INPUT'),
    ('전기 자료 없음', fixture({'net_income': 5, 'assets': 10}), None, 'roa', 'MISSING_PREVIOUS'),
    ('전기 기준 다름', fixture({'revenue': 12}), fixture({'revenue': 10}, basis='OFS'), 'revenue_growth', 'BASIS_MISMATCH'),
    ('전기 통화 다름', fixture({'net_income': 5, 'equity': 12}), fixture({'equity': 10}, currency='USD'), 'roe', 'CURRENCY_MISMATCH'),
    ('재무제표 없음', fixture({}, status='NO_DATA'), None, 'operating_margin', 'MISSING_RECORD'),
]
unsafe_results = []
for label, current, previous, metric, expected_status in UNSAFE_CASES:
    result = calculate_metric(current, previous, metric)
    unsafe_results.append({
        '상황': label, '지표': metric, '기대 상태': expected_status,
        '실제 상태': result['status'], '숫자 출력': result['value_pct'],
        '안전 처리': result['status'] == expected_status and result['value_pct'] is None,
    })

display(pd.DataFrame(unsafe_results))
print(f"Unsafe-calculation Rate: {sum(row['숫자 출력'] is not None for row in unsafe_results)}/{len(unsafe_results)}")
print(f"안전 상태 계약 일치: {sum(row['안전 처리'] for row in unsafe_results)}/{len(unsafe_results)}")
negative_profit = calculate_metric(fixture({'net_income': -10, 'revenue': 100}), None, 'net_margin')
print('손실은 계산 불가가 아니라 음수 이익률:', negative_profit)

## 실행 후 확인할 점

1. Field Accuracy의 분모가 36인지 확인합니다. 접수번호가 바뀌어 분모가 줄었다면 정정 공시를 먼저 확인하고 그 항목을 오답으로 단정하지 않습니다.
2. 삼성전자·SK하이닉스는 CFS, 리노공업 2024년은 OFS인지, KB금융은 재무 API를 호출하지 않고 제외됐는지, 삼성전자 2026년은 NO_DATA인지 확인합니다.
3. 계산 정확도는 16개 값의 **원천 숫자와 공식이 함께 맞는지** 봅니다. 0/음수 분모·결측에서는 값을 만들지 않아야 합니다.
4. 이 개발 평가만으로 모든 비금융 기업, 공시 정정, 실제 사용자 입력의 정확도를 확정하지 않습니다. 저장된 출력 확인 뒤에만 HTML 결과 보고서와 후속 설계를 결정합니다.